In [5]:
import torch
from pathlib import Path
import os
import sys
sys.path.append(os.path.abspath('../')) 
from src.utils.prepare_data import INRDataProcessor

# Setup the processor with your directories
processor = INRDataProcessor(
    input_root="data/mnist-inrs", 
    output_root="data/processed_inrs"
)

print("✅ Processor loaded.")

✅ Processor loaded.


In [7]:
sample_file = Path("data/mnist_png_training_0_1.pt")

# Pick the very first processed file to test
print(f"Checking file: {sample_file.name}\n" + "-"*40)

# Load the file from disk
data = torch.load(sample_file, map_location='cpu')
neurons = data['neurons']
layer_ids = data['layer_ids']

# Verify the matrix layout
print(f"Neurons container type: {type(neurons)}")
print(f"Neurons matrix shape:   {neurons.shape}  <- (Should be 2D: [Total_Neurons, 33])")
print(f"Layer IDs tensor shape: {layer_ids.shape}  <- (Should be 1D: [Total_Neurons])")
print(f"Layers found in file:   {torch.unique(layer_ids).tolist()}")

Checking file: mnist_png_training_0_1.pt
----------------------------------------
Neurons container type: <class 'torch.Tensor'>
Neurons matrix shape:   torch.Size([65, 33])  <- (Should be 2D: [Total_Neurons, 33])
Layer IDs tensor shape: torch.Size([65])  <- (Should be 1D: [Total_Neurons])
Layers found in file:   [0, 1, 2]


In [8]:
# Define where you want to save the reconstructed model checkpoint
output_pth_path = "data/reconstructed_sample.pth"

print("Attempting to reconstruct .pth file...")
print("-" * 40)

# Run the recovery function
processor.save_reconstructed_pth(
    processed_path=sample_file, 
    output_path=output_pth_path
)

# Verify the file was actually written to your drive
if Path(output_pth_path).exists():
    print(f"\n🎉 SUCCESS! Reconstructed file created at: {output_pth_path}")
else:
    print("\n❌ FAILED: The .pth file was not generated.")

Attempting to reconstruct .pth file...
----------------------------------------
Successfully reconstructed checkpoint to: data/reconstructed_sample.pth

🎉 SUCCESS! Reconstructed file created at: data/reconstructed_sample.pth


In [9]:
# Load the reconstructed model state dictionary
state_dict = torch.load(output_pth_path, map_location='cpu')

print("Reconstructed Model Layers:")
print("-" * 40)
for key, tensor in state_dict.items():
    print(f"Layer Key: {key:<15} | Shape: {list(tensor.shape)}")

# Quick structural sanity check
assert 'seq.0.weight' in state_dict, "Missing layer 0 weights!"
print("\n✅ All structural checks passed! Your data is fully recoverable.")

Reconstructed Model Layers:
----------------------------------------
Layer Key: seq.0.weight    | Shape: [32, 2]
Layer Key: seq.0.bias      | Shape: [32]
Layer Key: seq.1.weight    | Shape: [32, 32]
Layer Key: seq.1.bias      | Shape: [32]
Layer Key: seq.2.weight    | Shape: [1, 32]
Layer Key: seq.2.bias      | Shape: [1]

✅ All structural checks passed! Your data is fully recoverable.
